In [1]:
%cd /home

/home


In [2]:
import torch
from trainer import TrainingConfig, UNetTrainer
from utils import NpyImageDataset, channel_normalize
from diffusers import UNet2DModel
from diffusers.optimization import get_cosine_schedule_with_warmup

from trainer import UNetTrainer
from utils import add_noise

In [3]:
channel_mean = [0.1382167, 0.1816227]
channel_std  = [0.32978467, 0.51380478]

In [4]:
config = TrainingConfig()

dataset_train = NpyImageDataset(
    folder="/mnt/sciml/a.sadreev/sea_ice_data/valid",
    transform=lambda x: channel_normalize(x, channel_mean, channel_std),
    preload = False,
    mmap_mode = 'r',
)

train_dataloader = torch.utils.data.DataLoader(dataset_train, batch_size=config.train_batch_size, shuffle=True, num_workers = 6)

In [5]:
config = TrainingConfig()

dataset_valid = NpyImageDataset(
    folder="/mnt/sciml/a.sadreev/sea_ice_data/test",
    transform=lambda x: channel_normalize(x, channel_mean, channel_std),
    preload = False,
    mmap_mode = 'r',
)

valid_dataloader = torch.utils.data.DataLoader(dataset_valid, batch_size=config.train_batch_size, shuffle=True, num_workers = 6)

In [ ]:
from diffusers.models import UNetMotionModel

model = UNetMotionModel()

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=config.lr_warmup_steps,
    num_training_steps=(len(train_dataloader) * config.num_epochs),
)

trainer = UNetTrainer(config=config,
                       model=model, 
                       optimizer=optimizer, 
                       data_loader_train=train_dataloader, 
                       data_loader_val=valid_dataloader, 
                       lr_scheduler=lr_scheduler, 
                       add_noise_func=add_noise)

In [9]:
trainer.train_loop()

  0%|          | 0/365 [00:00<?, ?it/s]

TypeError: UNetMotionModel.forward() missing 1 required positional argument: 'encoder_hidden_states'